<a href="https://colab.research.google.com/github/danielbr9382/Corpus2GeoRAG_Pacific/blob/main/Processing/colab_ner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NER + DATE/TIME for the GeoRAG corpus
Upload `DATA.json` (or a JSON/JSONL file containing text records). The notebook runs the known BERT baseline for PER/ORG/LOC/MISC and spaCy for DATE/TIME. DATE/TIME are text mentions, not validated event dates.

Start with `LIMIT = 2`, inspect the outputs, then set `LIMIT = None` for the full corpus.


In [1]:
%pip -q install transformers spacy torch
!python -m spacy download en_core_web_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 69.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [1]:
from google.colab import files
uploaded = files.upload()
INPUT_PATH = next(iter(uploaded))
LIMIT = 2  # Set to None to process all documents


Saving DATA.json to DATA.json


In [2]:
"""Baseline mention extraction for the project's sample JSON and record-style JSON/JSONL.

Run in Colab after installing transformers, torch, spaCy and en_core_web_sm:
    python colab_ner.py DATA.json --output-dir ner_output --limit 3

The Hugging Face model is a baseline, not the project's final model. spaCy adds
DATE/TIME spans; these are temporal mentions, not verified event dates.
"""

from __future__ import annotations

import argparse
import csv
import hashlib
import json
import logging
import re
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterator

SCHEMA_VERSION = "0.1.0-example"
HF_MODEL = "dslim/bert-base-NER"
SPACY_MODEL = "en_core_web_sm"
STRIDE = 64
LOG = logging.getLogger("colab_ner")


def sha256_text(value: str) -> str:
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


def json_scalar(value: Any) -> Any:
    """Keep NumPy/Torch scalar scores numeric in the raw JSON output."""
    if hasattr(value, "item"):
        return value.item()
    return str(value)


def iter_documents(path: Path) -> Iterator[dict[str, Any]]:
    """Accept the observed title:text map and simple future record/list formats."""
    if path.suffix.lower() == ".jsonl":
        with path.open(encoding="utf-8-sig") as stream:
            records = (json.loads(line) for line in stream if line.strip())
            yield from _records_to_documents(records)
        return
    with path.open(encoding="utf-8-sig") as stream:
        data = json.load(stream)
    if isinstance(data, dict) and isinstance(data.get("text"), str):
        yield from _records_to_documents([data])
        return
    if isinstance(data, dict) and all(isinstance(v, str) for v in data.values()):
        for title, content in data.items():
            yield {
                "document_id": str(title),
                "article_title": str(title),
                "text": content,
                "source_record": {"article_title": title, "text": content},
            }
        return
    if isinstance(data, dict) and isinstance(data.get("documents"), list):
        data = data["documents"]
    if not isinstance(data, list):
        raise ValueError("Expected a title:text JSON map, a record/list JSON, or JSONL records")
    yield from _records_to_documents(data)


def _records_to_documents(records: Any) -> Iterator[dict[str, Any]]:
    for index, record in enumerate(records):
        if not isinstance(record, dict) or not isinstance(record.get("text"), str):
            raise ValueError(f"Record {index} must be an object with a string 'text'")
        title = record.get("article_title") or record.get("title")
        document_id = record.get("document_id") or record.get("article_id") or title
        if document_id is None:
            document_id = f"sha256:{sha256_text(record['text'])[:20]}"
        yield {
            "document_id": str(document_id),
            "article_title": title,
            "text": record["text"],
            "source_record": record,
        }


def iter_text_units(text: str) -> Iterator[tuple[int, int, int, str]]:
    """Split at blank lines, preserving exact document character positions."""
    left = 0
    index = 0
    for boundary in list(re.finditer(r"\n[ \t]*\n+", text)) + [None]:
        right = boundary.start() if boundary is not None else len(text)
        segment = text[left:right]
        if segment.strip():
            leading = len(segment) - len(segment.lstrip())
            trailing = len(segment.rstrip())
            start = left + leading
            end = left + trailing
            yield index, start, end, text[start:end]
            index += 1
        left = boundary.end() if boundary is not None else len(text)


def load_models() -> tuple[Any, Any, dict[str, Any]]:
    import spacy
    import torch
    import transformers
    from transformers import AutoModelForTokenClassification, AutoTokenizer, pipeline

    tokenizer = AutoTokenizer.from_pretrained(HF_MODEL, use_fast=True)
    if not tokenizer.is_fast:
        raise RuntimeError("A fast tokenizer is required for exact character offsets")
    model = AutoModelForTokenClassification.from_pretrained(HF_MODEL)
    ner = pipeline(
        "ner", model=model, tokenizer=tokenizer,
        aggregation_strategy="average", stride=STRIDE,
        device=0 if torch.cuda.is_available() else -1,
    )
    temporal = spacy.load(
        SPACY_MODEL,
        disable=["tagger", "parser", "attribute_ruler", "lemmatizer"],
    )
    metadata = {
        "name_model": HF_MODEL,
        "revision_model": getattr(model.config, "_commit_hash", None),
        "revision_tokenizer": tokenizer.init_kwargs.get("_commit_hash"),
        "temporal_model": SPACY_MODEL,
        "temporal_model_version": temporal.meta.get("version"),
        "library_versions": {
            "transformers": transformers.__version__,
            "torch": torch.__version__,
            "spacy": spacy.__version__,
        },
        "labels": sorted(set(model.config.id2label.values())) + ["DATE", "TIME"],
        "threshold": None,
        "window_size_tokens": tokenizer.model_max_length,
        "stride_tokens": STRIDE,
        "aggregation_strategy": "average",
        "prompt": None,
        "version_prompt": None,
    }
    return ner, temporal, metadata


def extract_unit(
    unit_text: str,
    document_text: str,
    unit_start: int,
    document_id: str,
    paragraph_id: str,
    ner: Any,
    temporal: Any,
    raw_id: str,
) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    raw: dict[str, Any] = {"ref_output_raw": raw_id, "hf": None, "spacy_temporal": None, "errors": []}
    candidates: list[tuple[str, dict[str, Any]]] = []
    try:
        predictions = ner(unit_text)
        raw["hf"] = [dict(p) for p in predictions]
        candidates.extend(("huggingface", dict(p)) for p in predictions)
    except Exception as exc:
        LOG.exception("Hugging Face extraction failed for %s", paragraph_id)
        raw["errors"].append({"extractor": "huggingface", "error": repr(exc)})
    try:
        predictions = [
            {"start": e.start_char, "end": e.end_char, "word": e.text,
             "entity_group": e.label_, "score": None}
            for e in temporal(unit_text).ents if e.label_ in {"DATE", "TIME"}
        ]
        raw["spacy_temporal"] = predictions
        candidates.extend(("spacy_temporal", p) for p in predictions)
    except Exception as exc:
        LOG.exception("Temporal extraction failed for %s", paragraph_id)
        raw["errors"].append({"extractor": "spacy_temporal", "error": repr(exc)})

    mentions: list[dict[str, Any]] = []
    seen: set[tuple[int, int, str, str]] = set()
    for extractor, prediction in candidates:
        start, end = prediction.get("start"), prediction.get("end")
        label = prediction.get("entity_group")
        key = (start, end, str(label), extractor)
        if key in seen:  # only the same prediction repeated by overlapping windows
            continue
        seen.add(key)
        valid_offsets = (
            isinstance(start, int) and isinstance(end, int)
            and 0 <= start < end <= len(unit_text)
            and document_text[unit_start + start:unit_start + end] == unit_text[start:end]
        )
        surface = unit_text[start:end] if valid_offsets else prediction.get("word")
        status = "valid" if valid_offsets and isinstance(label, str) else "invalid"
        mention_id = str(uuid.uuid5(
            uuid.NAMESPACE_URL,
            f"{document_id}|{sha256_text(document_text)}|{paragraph_id}|{extractor}|{start}|{end}|{label}",
        ))
        mentions.append({
            "mention_id": mention_id,
            "text": surface,
            "label": label,
            "label_source": extractor,
            "start": start,
            "end": end,
            "start_document": unit_start + start if valid_offsets else None,
            "end_document": unit_start + end if valid_offsets else None,
            "score": float(prediction["score"]) if prediction.get("score") is not None else None,
            "kind_score": "model_token_probability_aggregate" if extractor == "huggingface" else None,
            "status_validation": status,
            "validation_errors": [] if status == "valid" else ["Invalid span or missing label"],
            "ref_output_raw": raw_id,
        })
    mentions.sort(key=lambda m: (m["start"] if isinstance(m["start"], int) else -1, str(m["label"])))
    return mentions, raw


def run(input_path: str | Path, output_dir: str | Path = "ner_output", limit: int | None = None) -> dict[str, Any]:
    input_path, output_dir = Path(input_path), Path(output_dir)
    if not input_path.is_file():
        raise FileNotFoundError(input_path)
    if limit is not None and limit < 1:
        raise ValueError("limit must be positive")
    output_dir.mkdir(parents=True, exist_ok=True)
    ner, temporal, config = load_models()
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
    processed_at = datetime.now(timezone.utc).isoformat()
    config_hash = sha256_text(json.dumps(config, sort_keys=True, default=str))
    counts = {"documents": 0, "paragraphs": 0, "mentions": 0, "invalid_mentions": 0, "extractor_errors": 0}
    source_path = output_dir / f"{run_id}_sources.jsonl"
    mentions_path = output_dir / f"{run_id}_mentions.jsonl"
    raw_path = output_dir / f"{run_id}_raw.jsonl"
    csv_path = output_dir / f"{run_id}_mentions.csv"
    started = time.perf_counter()
    with source_path.open("w", encoding="utf-8") as source_file, \
         mentions_path.open("w", encoding="utf-8") as mention_file, \
         raw_path.open("w", encoding="utf-8") as raw_file, \
         csv_path.open("w", encoding="utf-8", newline="") as csv_file:
        csv_columns = ["document_id", "article_title", "paragraph_id", "label", "text", "start", "end", "start_document", "end_document", "score", "status_validation"]
        writer = csv.DictWriter(csv_file, fieldnames=csv_columns)
        writer.writeheader()
        for document in iter_documents(input_path):
            if limit is not None and counts["documents"] >= limit:
                break
            document_id = document["document_id"]
            document_text = document["text"]
            source_ref = f"{run_id}:document:{counts['documents']}"
            source_file.write(json.dumps({
                "source_ref": source_ref,
                "document_id": document_id,
                "source_record": document["source_record"],
                "text_hash": sha256_text(document_text),
            }, ensure_ascii=False) + "\n")
            counts["documents"] += 1
            for index, unit_start, unit_end, unit_text in iter_text_units(document_text):
                paragraph_id = f"{document_id}:p{index}"
                raw_id = f"{run_id}:raw:{counts['documents']}:{index}"
                unit_started = time.perf_counter()
                mentions, raw = extract_unit(
                    unit_text, document_text, unit_start, document_id,
                    paragraph_id, ner, temporal, raw_id,
                )
                raw_file.write(json.dumps(raw, ensure_ascii=False, default=json_scalar) + "\n")
                status = "error" if raw["errors"] else "success"
                record = {
                    "schema_version": SCHEMA_VERSION,
                    "run_id": run_id,
                    "source_ref": source_ref,
                    "document_id": document_id,
                    "article_title": document["article_title"],
                    "paragraph_id": paragraph_id,
                    "paragraph_index": index,
                    "text": unit_text,
                    "start_document": unit_start,
                    "end_document": unit_end,
                    "text_hash": sha256_text(unit_text),
                    "mentions": mentions,
                    "processing": {
                        "at_processed": processed_at,
                        "status": status,
                        "time_seconds": round(time.perf_counter() - unit_started, 3),
                        "config_hash": config_hash,
                        "ref_output_raw": raw_id,
                    },
                }
                mention_file.write(json.dumps(record, ensure_ascii=False) + "\n")
                for mention in mentions:
                    writer.writerow({
                        "document_id": document_id,
                        "article_title": document["article_title"],
                        "paragraph_id": paragraph_id,
                        **{k: mention[k] for k in ["label", "text", "start", "end", "start_document", "end_document", "score", "status_validation"]},
                    })
                counts["paragraphs"] += 1
                counts["mentions"] += len(mentions)
                counts["invalid_mentions"] += sum(m["status_validation"] == "invalid" for m in mentions)
                counts["extractor_errors"] += len(raw["errors"])
            LOG.info("Processed %d documents: %s", counts["documents"], document_id)
    manifest = {
        "schema_version": SCHEMA_VERSION,
        "run_id": run_id,
        "input_file": input_path.name,
        "input_sha256": hashlib.sha256(input_path.read_bytes()).hexdigest(),
        "processing": {"at_processed": processed_at, "time_seconds": round(time.perf_counter() - started, 3), "status": "complete_with_errors" if counts["extractor_errors"] else "complete"},
        "configuration": config,
        "config_hash": config_hash,
        "counts": counts,
        "files": {"sources": source_path.name, "mentions": mentions_path.name, "raw": raw_path.name, "csv": csv_path.name},
    }
    manifest_path = output_dir / f"{run_id}_manifest.json"
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    LOG.info("Wrote %s", manifest_path)
    return manifest


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("input_path", type=Path)
    parser.add_argument("--output-dir", type=Path, default=Path("ner_output"))
    parser.add_argument("--limit", type=int, default=None, help="Pilot on the first N documents")
    args = parser.parse_args()
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
    print(json.dumps(run(args.input_path, args.output_dir, args.limit), ensure_ascii=False, indent=2, default=str))




In [3]:
from pathlib import Path
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
manifest = run(INPUT_PATH, Path("ner_output"), limit=LIMIT)
print(json.dumps(manifest, ensure_ascii=False, indent=2, default=str))


config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  433MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{
  "schema_version": "0.1.0-example",
  "run_id": "20261005T145006Z-989e7b59",
  "input_file": "DATA.json",
  "input_sha256": "d58044161d4ec56f260852b9a773a9858b09438a97c00d8477c5dcc1101d71a2",
  "processing": {
    "at_processed": "2026-10-05T14:50:06.993347+00:00",
    "time_seconds": 21.873,
    "status": "complete"
  },
  "configuration": {
    "name_model": "dslim/bert-base-NER",
    "revision_model": "d1a3e8f13f8c3566299d95fcfc9a8d2382a9affc",
    "revision_tokenizer": null,
    "temporal_model": "en_core_web_sm",
    "temporal_model_version": "3.8.0",
    "library_versions": {
      "transformers": "5.17.0",
      "torch": "2.11.0+cpu",
      "spacy": "3.8.16"
    },
    "labels": [
      "B-LOC",
      "B-MISC",
      "B-ORG",
      "B-PER",
      "I-LOC",
      "I-MISC",
      "I-ORG",
      "I-PER",
      "O",
      "DATE",
      "TIME"
    ],
    "threshold": null,
    "window_size_tokens": 512,
    "stride_tokens": 64,
    "aggregation_strategy": "average",
    "prompt": n

In [4]:
import shutil
archive = shutil.make_archive("ner_output", "zip", "ner_output")
files.download(archive)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>